In [2]:
import pickle, time, sys, os, tempfile
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, HTML

import opendss.tools.dss_converter as odc
from multiconductor.shortcircuit.calc_sc import calc_sc as mc_calc_sc
from opendss.scd.short_circuit import calc_sc as dss_calc_sc

# ── CYME setup ──────────────────────────────────────────────────────────────
cyme_python_path = os.environ.get("CYME_PYTHON_PATH", r"C:\CYME\CYME")
if cyme_python_path not in sys.path:
    sys.path.insert(0, cyme_python_path)
import cympy
from cyme.scd.short_circuit import calc_sc as cyme_calc_sc

# ── Use absolute paths (DSS changes CWD) ───────────────────────────────────
src_dir  = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\mc_xfmr")
cyme_dir = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\cyme")
pkl_files = sorted(src_dir.glob("*.pkl"))
total = len(pkl_files)

FAULT = "3ph"

# ── Progress display helper ─────────────────────────────────────────────────
def _progress_bar(i, total, circuit, elapsed, eta, status="OK"):
    pct = i / total * 100
    bar_len = 30
    filled = int(bar_len * i // total)
    bar = "█" * filled + "░" * (bar_len - filled)
    eta_m, eta_s = divmod(int(eta), 60)
    html = (
        f"<b>[{i}/{total}]</b> {bar} {pct:.0f}% &nbsp;│&nbsp; "
        f"<code>{circuit}</code> ({elapsed:.1f}s) &nbsp;│&nbsp; "
        f"ETA {eta_m}m{eta_s:02d}s &nbsp;│&nbsp; {status}"
    )
    return html

progress_handle = display(HTML("Starting…"), display_id=True)

rows = []
t_start = time.time()
n_ok, n_fail = 0, 0

for i, pkl_path in enumerate(pkl_files, 1):
    circuit_key = pkl_path.stem
    #print(f"Processing {circuit_key} ({i}/{total})…")
    t0 = time.time()

    try:
        with open(pkl_path, "rb") as fh:
            net = pickle.load(fh)
        #print("pickle loaded, running MC short circuit…")
        # ── MC short circuit ────────────────────────────────────────────
        mc_calc_sc(net, fault=FAULT, case="max")
        mc_ikss = pd.to_numeric(net.res_bus_sc["ikss_ka"], errors="coerce").dropna()
        #print("running dss short circuit…")
        # ── DSS short circuit ───────────────────────────────────────────
        #dss_file = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\dss\{circuit_key}.dss"
        #odc.mc_net_to_opendss(net, str(dss_file))
        #dss_result = dss_calc_sc(str(dss_file), fault=FAULT)
        #dss_ikss = pd.to_numeric(dss_result["res_bus_sc"]["ikss_ka"], errors="coerce").dropna()
        #print("running cyme short circuit…")
        # ── CYME short circuit ──────────────────────────────────────────
        cyme_ikss_mean = np.nan
        sxst_path = cyme_dir / f"{circuit_key}.sxst"
        if sxst_path.exists():
            cyme_result = cyme_calc_sc(str(sxst_path), fault=FAULT, close_on_finish=True)
            cyme_ikss = pd.to_numeric(cyme_result["res_bus_sc"]["ikss_ka"], errors="coerce").dropna()
            cyme_ikss_mean = cyme_ikss.mean() if not cyme_ikss.empty else np.nan

        rows.append({
            "CKT_KEY":       circuit_key,
            "MC_IKSS_MEAN":  mc_ikss.mean()  if not mc_ikss.empty  else np.nan,
            "MC_IKSS_MAX":   mc_ikss.max()   if not mc_ikss.empty  else np.nan,
            "MC_N_BUSES":    len(mc_ikss),
            #"DSS_IKSS_MEAN": dss_ikss.mean() if not dss_ikss.empty else np.nan,
            #"DSS_IKSS_MAX":  dss_ikss.max()  if not dss_ikss.empty else np.nan,
            #"DSS_N_BUSES":   len(dss_ikss),
            "CYME_IKSS_MEAN": cyme_ikss_mean,
            "CYME_IKSS_MAX":  cyme_ikss.max() if sxst_path.exists() and not cyme_ikss.empty else np.nan,
            "CYME_N_BUSES":   len(cyme_ikss) if sxst_path.exists() and not cyme_ikss.empty else 0,
        })

        n_ok += 1
        elapsed = time.time() - t0
        total_elapsed = time.time() - t_start
        avg = total_elapsed / i
        eta = avg * (total - i)
        progress_handle.update(HTML(_progress_bar(i, total, circuit_key, elapsed, eta, "✓")))

    except Exception as e:
        print(f"Error processing {circuit_key}: {e}")
        n_fail += 1
        elapsed = time.time() - t0
        total_elapsed = time.time() - t_start
        avg = total_elapsed / i
        eta = avg * (total - i)
        progress_handle.update(HTML(_progress_bar(i, total, circuit_key, elapsed, eta, f"✗ {e}")))
        try:
            cympy.study.Close(AskForSave=False)
        except Exception:
            pass

total_elapsed = time.time() - t_start
df_sc_comparison = pd.DataFrame(rows)
progress_handle.update(HTML(
    f"<b>Done in {total_elapsed:.1f}s — {n_ok} OK, {n_fail} failed, "
    f"{len(df_sc_comparison)} circuits.</b>"
))
df_sc_comparison

KeyboardInterrupt: 

In [ ]:
# ── Summary statistics & pairwise comparisons ──────────────────────────────
df = df_sc_comparison.copy()

# Percent differences (MC as reference)
df["MC_vs_DSS_pct"]  = ((df["DSS_IKSS_MEAN"]  - df["MC_IKSS_MEAN"]) / df["MC_IKSS_MEAN"] * 100)
df["MC_vs_CYME_pct"] = ((df["CYME_IKSS_MEAN"] - df["MC_IKSS_MEAN"]) / df["MC_IKSS_MEAN"] * 100)

mask_all = df[["MC_IKSS_MEAN", "DSS_IKSS_MEAN", "CYME_IKSS_MEAN"]].notna().all(axis=1)
df_all = df[mask_all]

print(f"Circuits with all three engines: {len(df_all)} / {len(df)}")
print(f"\nMean ikss_ka:")
print(f"  MC:   {df_all['MC_IKSS_MEAN'].mean():.4f}")
print(f"  DSS:  {df_all['DSS_IKSS_MEAN'].mean():.4f}")
print(f"  CYME: {df_all['CYME_IKSS_MEAN'].mean():.4f}")
print(f"\nMean absolute % difference (MC reference):")
print(f"  MC vs DSS:  {df_all['MC_vs_DSS_pct'].abs().mean():.2f}%")
print(f"  MC vs CYME: {df_all['MC_vs_CYME_pct'].abs().mean():.2f}%")

df.describe()

In [ ]:
# ── Save results ────────────────────────────────────────────────────────────
out_path = Path(r"c:\Users\fgonzales\git\mc_opendss\validation\scd_comparison_results.csv")
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")